In [2]:
#Library
import pandas as pd
import matplotlib.pyplot as plt
import spacy
from collections import Counter
import sys
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import FunctionTransformer
from preprocessing import preprocessing


from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder



from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

spacy.cli.download("en_core_web_md")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 95.0 MB/s  0:00:00 eta 0:00:01


✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


  Using cached en_core_web_md-3.8.0-py3-none-any.whl (33.5 MB)
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [3]:
df_train=pd.read_csv("data/train.csv")
df_test=pd.read_csv("data/test.csv")
df_train=df_train.drop(columns=['id'])
df_train=df_train.drop_duplicates()

In [4]:
ponctuation = [".", "!", "?"]
preprocesser=FunctionTransformer(preprocessing)

In [ ]:
X_train=df_train.drop(columns="target")
y_train=df_train["target"]

In [7]:
columns=["has_.","has_?","has_!","has_url","has_CAP","url_is_https"]

features = ColumnTransformer([
    ("txt", TfidfVectorizer(), "text"),                           # LE texte → TF-IDF
    ("kw",  OneHotEncoder(handle_unknown="ignore"), ["keyword"]), # catégoriel → one-hot
    ("binaire", "passthrough", columns),                          # binaire → tel quel
])

pipe = Pipeline([
    ("preprocesser", preprocesser),   # nettoyage seulement
    ("features", features),             # remplace l'étape "vect"
    ("clf", LogisticRegression(max_iter=1000)),
])


param_grid = [
    {"features": [CountVectorizer(), TfidfVectorizer()],
     "features__ngram_range": [(1, 1), (1, 2)],
     "features__min_df": [1, 5]},
]

In [8]:
pipe.fit(X_train.head(50), y_train.head(50))

ValueError: y should be a 1d array, got an array of shape (50, 4) instead.

In [ ]:
grid = GridSearchCV(pipe, param_grid, cv=5, scoring="f1_macro")

In [ ]:
grid.fit(df_train)